In [ ]:
import optuna
import torch
from PAF_model import *
import pandas as pd
import numpy as np
import json
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Optuna wrapper function (note: not currently put in k-fold validation format, but that can be arranged within each objective function)

In [ ]:
# ----- Main BO Process -----
def optuna_study(objective, outname, n_trials = 50, n_jobs = 1):
    np.random.seed(42)
    torch.manual_seed(42)
    study = optuna.create_study(direction="minimize",
                                pruner=optuna.pruners.PercentilePruner(
                                    n_startup_trials=5,      # Need 5 finished trials to know what "top 25%" means
                                    n_warmup_steps=15,      
                                    interval_steps=5 ,
                                    percentile = 25.0)) 
    try:
        study.optimize(objective, n_trials=n_trials, n_jobs=n_jobs)
        print("########\n\n\n\n########") 
        print(study.best_params)
    except KeyboardInterrupt:
        print("Optimization stopped")
        print(f"Best so far: {study.best_params}")
        
    df = study.trials_dataframe()
    df.to_csv(outname + ".csv", index=False)

    # Save best parameters as JSON
    best_params = {
        "best_params": study.best_params,
        "best_value": study.best_value,
        "best_trial": study.best_trial.number
    }
    with open(outname + ".json", "w") as f:
        json.dump(best_params, f, indent=2)
    print(f'Best hyperparameters json file saved as {outname}.json')

# GAT embeddings model training

In [ ]:
def GAT_objective(trial, epochs = 50):
    hidden_dim = trial.suggest_categorical("hidden_dim", [32,64,128,256,512])
    embed_dim = trial.suggest_categorical("embed_dim", [16, 32, 64,128,256])
    weight_decay = trial.suggest_float("weight_decay",5e-4,5e-2, log=True )    
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    trainer = PAF_trainer(
                 x_esm_embed = input_data.shape[1],
                 hidden_dim= hidden_dim,
                 output_dim= embed_dim,
                 lr = [lr, 0,0],
                 l2_coef = [weight_decay, 0,0],
                 dropout = [dropout, 0 , 0],
                 device = device)

    pbar = tqdm.tqdm(range(1, epochs + 1),
                            desc="GAT Training", unit="epoch")
    for epoch in pbar:
        _ = trainer.train_embeddings(input_data, train_edges, train_weights,train_val_edges,train_val_edge_weights,
                                        epochs= 1,verbose=False)
        pbar.set_postfix({
                        "Train" : f"{trainer.train_CL_history[-1]:.4f}",
                        "Valid":  f"{trainer.valid_CL_history[-1]:.4f}",
                        "Best-Valid": f"{trainer.best_CL:.4f}"
                    })
        trial.report(loss, step=epoch)
        if trial.should_prune():
            #print(f"Trial {trial.number} is drifting. Pruning at epoch {epoch}.")
            raise optuna.exceptions.TrialPruned()
        if trainer.train_CL_history[-1] is np.nan :
            #print(f"✂️ Trial {trial.number} diverged with loss {pipeline.trainer.train_losses[-1]:.2f}. Pruning.")
            raise optuna.exceptions.TrialPruned()

    loss = trainer.valid_step_prediction(input_data, train_val_edges,train_val_edge_weights)
    return loss


In [ ]:
optuna_study(GAT_objective, 'GAT_optuna_study', n_trials = 50, n_jobs = 1)

with open("GAT_optuna_study.json", "r") as f:
    best_params = json.load(f)
    
best_params = best_params['best_params']

trainer = PAF_trainer(
                x_esm_embed = input_data.shape[1],
                hidden_dim= best_params['hidden_dim'],
                output_dim= best_params['embed_dim'],
                lr = [best_params['lr'], 0,0],
                l2_coef = [best_params['weight_decay'], 0,0],
                dropout = [best_params['dropout'], 0 , 0],
                device = device)
cl = trainer.train_embeddings(input_data, train_edges, train_weights,train_val_edges,train_val_edge_weights,
                              epochs= 50,verbose=True)
trainer.save('GAT','GAT_model_optuna.pth')

# MLP model training

In [ ]:
def MLP_objective(trial, epochs = 50):
    weight_decay = trial.suggest_float("weight_decay",5e-4,5e-2, log=True )    
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    trainer = PAF_trainer(
                x_esm_embed = input_data.shape[1],
                hidden_dim= best_params['hidden_dim'],
                output_dim= best_params['embed_dim'],
                lr = [best_params['lr'], lr,0],
                l2_coef = [best_params['weight_decay'], weight_decay,0],
                dropout = [best_params['dropout'], dropout , 0],
                device = device)
    trainer.load("GAT",'GAT_model_optuna.pth')
    
    pbar = tqdm.tqdm(range(1, epochs + 1),
                            desc="MLP Training", unit="epoch")
    for epoch in pbar:
        _ = trainer.train_predictor(x_esm_embed, train_edges, train_weights,
                                    train_val_edges,train_val_edge_weights,
                                        epochs= 1,verbose=False)
        pbar.set_postfix({
                        "Train" : f"{trainer.train_MSE_history[-1]:.4f}",
                        "Valid":  f"{trainer.valid_MSE_history[-1]:.4f}",
                        "Best-Valid": f"{trainer.best_MSE:.4f}"
                    })
        trial.report(loss, step=epoch)
        if trial.should_prune():
            #print(f"Trial {trial.number} is drifting. Pruning at epoch {epoch}.")
            raise optuna.exceptions.TrialPruned()
        if trainer.train_MSE_history[-1] is np.nan :
            #print(f"✂️ Trial {trial.number} diverged with loss {pipeline.trainer.train_losses[-1]:.2f}. Pruning.")
            raise optuna.exceptions.TrialPruned()

    loss = trainer.valid_step_embeddings(x_esm_embed, train_val_edges,train_val_edge_weights)
    return loss


In [ ]:
optuna_study(MLP_objective, 'MLP_optuna_study', n_trials = 50, n_jobs = 1)

with open("MLP_optuna_study.json", "r") as f:
    best_params2 = json.load(f)
    
best_params2 = best_params2['best_params']

trainer = PAF_trainer(
                x_esm_embed = input_data.shape[1],
                hidden_dim= best_params['hidden_dim'],
                output_dim= best_params['embed_dim'],
                lr = [best_params['lr'], best_params['lr'],0],
                l2_coef = [best_params['weight_decay'], best_params['weight_decay'],0],
                dropout = [best_params['dropout'], best_params['dropout'] , 0],
                device = device)
trainer.load("GAT",'GAT_model_optuna.pth')
    
mse = trainer.train_predictor(input_data, train_edges, train_weights,train_val_edges,train_val_edge_weights,
                              epochs= 50,verbose=True)
trainer.save('MLP','MLP_model_optuna.pth')

# Flow model training
# NOTE: PERCENTILE IS NOT A HYPERPARAMETER TO TUNE BUT RATHER ARBITRARY THRESHOLD FOR A CONFIDENT INTERACTION

In [ ]:
def Flow_objective(trial, epochs = 50):
    weight_decay = trial.suggest_float("weight_decay",5e-4,5e-2, log=True )    
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    n_layers = trial.suggest_categorical("n_layers", [2,3,4])
    n_flows = trial.suggest_categorical("n_flows", [4, 5,6, 7,8])
    trainer = PAF_trainer(
                x_esm_embed = input_data.shape[1],
                hidden_dim= best_params['hidden_dim'],
                output_dim= best_params['embed_dim'],
                n_layers = n_layers,
                n_flows = n_flows,
                percentile= 0.5,
                lr = [best_params['lr'], best_params2['lr'],lr],
                l2_coef = [best_params['weight_decay'],
                           best_params2['weight_decay'],
                           weight_decay],
                dropout = [best_params['dropout'],
                           best_params2['dropout'],
                           dropout],
                device = device)
    trainer.load("GAT",'GAT_model_optuna.pth')
    trainer.load("MLP",'MLP_model_optuna.pth')
    
    pbar = tqdm.tqdm(range(1, epochs + 1),
                            desc="NVP Training", unit="epoch")
    for epoch in pbar:
        _ = trainer.train_flow(x_esm_embed,
                               train_edges, train_weights,
                                train_val_edges,train_val_edge_weights,
                                epochs= 1,verbose=False)
        pbar.set_postfix({
                        "Train" : f"{trainer.train_NLL_history[-1]:.4f}",
                        "Valid":  f"{trainer.valid_NLL_history[-1]:.4f}",
                        "Best-Valid": f"{trainer.best_NLL:.4f}"
                    })
        trial.report(loss, step=epoch)
        if trial.should_prune():
            #print(f"Trial {trial.number} is drifting. Pruning at epoch {epoch}.")
            raise optuna.exceptions.TrialPruned()
        if trainer.train_NLL_history[-1] is np.nan :
            #print(f"✂️ Trial {trial.number} diverged with loss {pipeline.trainer.train_losses[-1]:.2f}. Pruning.")
            raise optuna.exceptions.TrialPruned()

    loss = trainer.valid_step_flow(x_esm_embed, train_val_edges,train_val_edge_weights)
    return loss

In [ ]:
optuna_study(Flow_objective, 'Flow_optuna_study', n_trials = 50, n_jobs = 1)

with open("Flow_optuna_study.json", "r") as f:
    best_params3 = json.load(f)
    
best_params3 = best_params3['best_params']

trainer = PAF_trainer(
            x_esm_embed = input_data.shape[1],
            hidden_dim= best_params['hidden_dim'],
            output_dim= best_params['embed_dim'],
            n_layers = best_params3['n_layers'],
            n_flows = best_params3['n_flows'],
            lr = [best_params['lr'],
                  best_params2['lr'],
                  best_params3['lr']],
            l2_coef = [best_params['weight_decay'],
                        best_params2['weight_decay'],
                        best_params3['weight_decay']],
            dropout = [best_params['dropout'],
                        best_params2['dropout'],
                        best_params3['dropout']],
            device = device)
cl = trainer.train_flow(input_data, train_edges, train_weights,train_val_edges,train_val_edge_weights,
                              epochs= 50,verbose=True)
trainer.save('RealNVP','Flow_model_optuna.pth')